# Drone Detection & Tracking — end to end on Kaggle T4×2

Trains **both** detector families in the matrix — **YOLO11n** and **RT-DETR-x2** — on
**Drone-vs-Bird (`dvb`)**, across both GPUs, and evaluates each.

This is the whole ML pipeline in one run: acquire → convert → split → build → verify →
train → score. Nothing in this repository downloads or trains by itself — every stage
below is an explicit command, and `sanity` is a hard gate that stops the run before it can
waste GPU hours on a leaked split.

### Why `dvb`, and why alone

`dvb` is the **only** dataset in the matrix that can *falsify* a false-positive claim.
Three of the four sources are drone-only tracking benchmarks: they contain no bird
annotations at all, so a detector that labelled every bird as a drone would score
perfectly on them. `dvb` has genuine bird negatives beside the drone class, which makes its
precision number *mean* something. It is also the dataset the deployed system exists to get
right — the failure mode it prevents is a bird being called a drone, and a page being raised
for nothing.

### Why both models

The matrix is 2 families × 7 combos, and the two families answer different questions.
YOLO11n (2.6 M params) is the **edge candidate** — the one small enough to sit behind
`nvinfer` on a Jetson with a ~3 ms budget. RT-DETR-x2 (42.3 M) is the **accuracy
reference**: how much accuracy is actually available in this data. Running only the small
model tells you what you could deploy; running both tells you what you are leaving on the
table.

They are not two settings of the same model, either. RT-DETR is **NMS-free** — its decoder
emits a fixed, already-deduplicated set of queries — so the NMS stage that YOLO needs is
absent rather than tuned. That is why the same tracker stack is legitimate for both: the
boxes arrive through the same `.boxes` attribute either way.

### What you should see

| Stage | Command | Approx. cost |
| --- | --- | --- |
| Verify | `anti-uav verify-env` | seconds |
| Weights | `anti-uav fetch-weights` | ~6 MB + ~130 MB |
| Data | `anti-uav download --dataset dvb` | ~6.7 GB |
| Preprocess | `convert` → `splits` → `build` → `sanity` | ~15–25 min |
| Train YOLO11n | `--model yolo11n --epochs 120` | **1.5–3 h** |
| Train RT-DETR-x2 | `--model rtdetr_x2 --epochs 100` | **4–6 h** |
| Evaluate | `anti-uav evaluate --cross` | ~2 min each |

Both fits Kaggle's 12-hour session, but together they leave little margin. **If you only
want the edge candidate, stop after the YOLO11n run** — section 7 explains how to read the
comparison if you have both.

---

## Before you run

**Settings → Accelerator → GPU T4 x2.** Kaggle restarts the session when you change this,
so do it before running anything.

**Settings → Internet → On.** Required three times over: the `git clone`, the `uv` install,
and the Kaggle dataset download.

**Add two secrets** (Settings → Secrets, or Add-ons → Secrets):

| Secret | Value |
| --- | --- |
| `KAGGLE_USERNAME` | your Kaggle account name |
| `KAGGLE_KEY` | your API key, from Account → Settings → API |

The notebook has no credentials of its own and cannot reach a private dataset without
these. They are read from secrets in the download step and never written into a file that
gets committed.

One Kaggle quirk worth knowing in advance: you may also need to **accept the rules** for
`romsham/dronevsbird-foryolo` in a browser tab. Kaggle gates that dataset behind a rules
acceptance that the API cannot satisfy for you, and the error it returns is a 403 with no
explanation.

---

In [ ]:
# --- 1. Clone -------------------------------------------------------------

REPO_URL = "https://github.com/uzairlol/drone-detection-tracking.git"
REPO_DIR = "/kaggle/working/anti-uav"

!git clone --depth 1 {REPO_URL} {REPO_DIR}

In [ ]:
# --- 2. Install -----------------------------------------------------------

# `uv` rather than `pip`: same resolution, a fraction of the wall clock, and it leaves an
# already-satisfied requirement alone. That last part is the point - Kaggle ships a
# CUDA-matched torch/numpy pair, and replacing it with a different build is the classic way
# to break `import torch` on a GPU that works fine.
#
# `--system` installs into the image's interpreter. A venv would need its bin/ prepended to
# PATH in every subsequent cell, which is one forgotten cell away from a silent ImportError.
#
# `pip install -e ".[train,download]"` is an exact substitute if you would rather not add
# a dependency manager; it is only slower.

!pip install --quiet uv
!uv pip install --system -e "{REPO_DIR}[train,download]"

## 3. Redirect scratch, then verify the environment

Everything writable goes to `/kaggle/tmp`, never `/kaggle/working`.

This is a hard limit, not a preference: only **20 GB** of `/kaggle/working` is autosaved per
notebook version, while the raw dataset plus the interim and processed trees come to ~15 GB
and then some. `/kaggle/tmp` is scratch — far larger, and discarded with the session.

`ANTI_UAV_DATA_DIR` moves the `data/` and `artifacts/` trees while `configs/` is still read
from the clone, so not one pipeline behaviour changes. It must be set **before** the first
`anti-uav` call, because path resolution is cached on first use.

Then read `verify-env` rather than trusting the exit code alone. Three lines carry the
meaning:

| Line | Must say | If it does not |
| --- | --- | --- |
| `profile` | `turing` | `pascal` means auto-detection did not run and **AMP is off** |
| `arch-kernels` | `[ok] … sm_75 present` | Kaggle's torch lacks Turing kernels; nothing else will help |
| `base-weights` | warn, not fail | expected — `*.pt` is gitignored; cell 5 fetches it |

This step exists because the failure it catches is silent. A torch wheel built without your
card's architecture imports cleanly, reports `torch.cuda.is_available() == True`, and then
dies hours in with `no kernel image is available for execution on the device`. Checking
availability is not enough; the arch list has to be checked too.

In [ ]:
import os

os.environ["ANTI_UAV_DATA_DIR"] = "/kaggle/tmp/anti-uav"

# Auto-detect is correct on a T4 now (sm_75 -> turing). Pinned anyway, so the notebook cannot
# silently pick a different profile if the accelerator is switched to a P100 (sm_60).
os.environ["ANTI_UAV_PROFILE"] = "turing"

!anti-uav verify-env

## 4. Acquire

**Weights first, on purpose.** ultralytics downloads a missing base checkpoint from *inside*
the trainer, so on a slow or blocked network the run dies *after* the dataset is built and the
run directory created. Fetching here moves that failure to a five-second one.

Then `dvb`, from the public Kaggle mirror. `--dry-run` first every time: it prints exactly
what would be fetched, from where, and how large — and it is the cheapest way to catch a
missing credential before committing to a 6.7 GB transfer.

In [ ]:
# Secrets: KAGGLE_USERNAME and KAGGLE_KEY. Never hard-coded, never written to disk.
try:
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secrets.get_secret("KAGGLE_KEY")
    print("credentials loaded from secrets")
except Exception as exc:
    print(f"no Kaggle secrets ({type(exc).__name__}) - cell 6 will report needs_credentials")

!anti-uav fetch-weights            # yolo11n.pt (~6 MB) + rtdetr-x2.pt (~130 MB)

In [ ]:
!anti-uav download --dataset dvb --dry-run

In [ ]:
!anti-uav download --dataset dvb

## 5. Preprocess — and why the order is not negotiable

```
convert  →  splits  →  build  →  sanity
```

`build` reads each frame's split field off the interim index. Build before splitting and you
materialise an **empty train set**, then `sanity` fails — correctly, but after wasting the
copy.

**`splits` assigns whole sequences, never individual frames.** All four datasets here are
video-derived, so adjacent frames are near-identical. Split at the frame level and the same
target's near-duplicates land on both sides of the boundary: mAP inflates by double digits and
the number is meaningless. `sanity` fails the build if any sequence appears in two splits, so
this is enforced rather than merely documented.

**`convert` reads in place.** The ~6.7 GB of PNGs is never copied into the working tree.

### What `stats` will tell you

Expect roughly **17.9k frames**, ~1.15 boxes each, and a bird-to-drone box ratio near 1.3.

The converter checks that ratio against the published statistics and **aborts** above 1.5.
This is not paranoia: the Kaggle mirror renames classes to bare integers, and a silent 0/1
inversion is invisible in a per-class mAP table unless you already know which way round it
should be — while inverting the one output the whole system depends on.

One surprise to expect: **`dvb` is configured as tiled at 640 px**, because its median target
(28 px) is below the 40 px threshold. But this mirror already ships 640×640 crops, so
`tile_grid` returns exactly one tile per image and tiling is a no-op. That is why this run is
~17.9k frames rather than the 4–6× the docs quote for a 4K source. The tiling path is still
exercised — it simply has nothing to do.

In [ ]:
!anti-uav convert --dataset dvb
!anti-uav stats   --dataset dvb

In [ ]:
# convert -> splits -> build -> sanity. The order is load-bearing: `build` reads each
# frame's split field off the interim index, so building first gives an empty train set.
# `sanity` exits non-zero on failure - read its output, do not just scroll past it.
!anti-uav splits --combo dvb
!anti-uav build  --combo dvb
!anti-uav sanity --combo dvb

## 6. Train

Dry-run first, every time. It prints the resolved hyperparameters, the output path and the
dataset summary without allocating a GPU — the cheapest possible way to catch a wrong combo,
and the only place the resolved batch is visible before it either works or OOMs.

### The two models do not get the same batch

| | YOLO11n | RT-DETR-x2 |
| --- | --- | --- |
| params | 2.6 M | 42.3 M |
| batch on T4×2 | **32** | **16** |
| per rank | 16 | 8 |
| epochs | 120 | 100 |
| mosaic | on | **off** |
| optimizer | SGD, lr 0.01 | AdamW, lr 0.0002 |
| NMS | yes | **no — NMS-free** |

RT-DETR gets half the batch because its hybrid encoder and 100 denoising queries keep far
more activations live than YOLO11n's backbone — memory scales with the query count, not with
the parameter count. **If you OOM, halve the batch before changing anything else.**

One number per GPU profile is not enough for that, so `configs/train/overrides/*.yaml`
carries both: a `force_batch` for the CNN and `force_batch_by_model` for the transformer.
`matrix list` resolves both correctly, which is the point — a planner that prints a batch the
run cannot allocate is worse than no planner.

### Reading the batch under DDP

The printed batch is the **total across both GPUs**, so 32 means 16 per rank. It is a fixed
integer rather than AutoBatch for two reasons: ultralytics resolves `batch: -1` to a
*fraction* of free VRAM and then raises outright when `world_size > 1`, and DDP splits the
total evenly, so an odd number would leave a rank idle.

### Performance expectation

| Run | Wall clock on T4×2 |
| --- | --- |
| YOLO11n, 120 epochs | 1.5–3 h |
| RT-DETR-x2, 100 epochs | 4–6 h |

Expect a **sub-2× speedup from two T4s**, closer to 1.8× on RT-DETR (a T4 at batch 8/rank is
far from saturated). These are order-of-magnitude estimates from FLOP counts, not
measurements. This is a good box for proving the pipeline end to end; the full 14-run matrix
belongs on the lab's RTX 4090/5090.

In [ ]:
!anti-uav train --model yolo11n  --combo dvb --profile turing --device 0,1 --workers 2 --dry-run
!anti-uav train --model rtdetr_x2 --combo dvb --profile turing --device 0,1 --workers 2 --dry-run

### 6a · YOLO11n — the edge candidate

**Stop after this cell** if you only want the model you could actually deploy. Everything
downstream works with this run alone.

In [ ]:
# ~1.5-3 h for 120 epochs. Add --epochs 30 for a fast pass.
#
# FALLBACK: if DDP cannot spawn inside the notebook kernel, re-run with --device 0
# (single GPU, same batch). Identical results, similar wall clock, no subprocess to
# go wrong. The preflight warns before training if you ask for more GPUs than are present.
!anti-uav train --model yolo11n --combo dvb --profile turing --device 0,1 --workers 2

### 6b · RT-DETR-x2 — the accuracy reference

**Optional, and the expensive half.** ~4–6 h, which roughly doubles the session. Run it only
if you have budget for both, because a comparison is the entire point — 120-epoch YOLO11n
against a truncated RT-DETR proves nothing about which architecture is better.

Its recipe differs from YOLO11n in exactly three places, and each has a reason:

1. **Mosaic off.** RT-DETR's hybrid encoder and denoising queries were designed for clean
   single-scale training. Leaving mosaic on costs accuracy *and* memory.
2. **Base LR 100× lower** (0.0002 vs 0.01), with AdamW. A nano-model SGD learning rate on a
   transformer either diverges or stalls outright.
3. **Warmup 1 epoch** instead of 3. This is fine-tuning from COCO, not training from scratch.

In [ ]:
# Resolves to batch 16 from force_batch_by_model - do not pass --batch, or you will
# override the value that was chosen for this card.
!anti-uav train --model rtdetr_x2 --combo dvb --profile turing --device 0,1 --workers 2

### Comparing the two

`list-runs` reads the run directories off disk and prints both side by side — run, model,
epochs, mAP50, mAP50-95, precision, recall and wall clock. No parsing needed.

Two things to hold onto when reading it:

- **mAP50-95 is the headline, not mAP50.** At a 28 px median target, mAP50 is generous — a
  detector that puts a loose box on the right patch scores well. The stricter metric is the
  one that discriminates.
- **The gap you should expect is small on `dvb`, and that is informative.** RT-DETR's
  advantage is on small, densely packed targets; this mirror ships 640×640 crops where the
  median target is already 28 px and tiling is a no-op. `dvb` is not the regime where a
  42 M transformer earns its cost. The interesting comparison is on `mmuav` at 12 px, tiled —
  which needs the Baidu-Pan transfer.

So the honest reading of a small gap here is *"the small model is not leaving accuracy on the
table for this dataset"*, which is a genuinely useful result: it means the deployable option is
not paying for 16× the parameters.

## 7. Evaluate

`--cross` is the number worth showing. A single combined mAP across heterogeneous datasets is
close to meaningless, because the sources disagree about what a hard sample is — MM-UAV's
12 px targets dominate the loss while Drone-vs-Bird's birds dominate the precision. The
per-dataset table is the honest one.

### Read the falsifiability note, not just the number

The evaluator prints a caveat next to any precision figure whose dataset cannot support the
claim:

> for `antiuav` and `mmuav`, precision cannot falsify a false-positive claim — neither
> contains bird annotations, so a detector that called every bird a drone would score
> perfectly.

On `dvb` it **can**. That is the entire reason this dataset is the demo, and it is the one
number in this notebook that can actually be *wrong* in an informative way.

One caveat specific to this run: `dvb` has **no negative-only frames** — every frame contains
at least one labelled object. A detector trained mostly on positives never learns a usable
confidence floor, so `rules.confidence.initiate` has nothing calibrated against it. The
`stats` output flags this. It is a property of the source data, not of the pipeline.

In [ ]:
# Both runs, same evaluator, same split files. Swap the --run to compare.
!anti-uav evaluate --run artifacts/runs/yolo11n/dvb --cross
!anti-uav evaluate --run artifacts/runs/rtdetr_x2/dvb --cross

## 8. Why `track-eval` refuses this dataset

Run it and it declines. **That is correct behaviour, and worth demonstrating rather than
working around** — it is the tooling refusing to report a number it cannot stand behind.

| Dataset | Bird negatives | MOT ground truth | Multi-object identity | Verdict |
| --- | --- | --- | --- | --- |
| `dvb` | **yes** | no | no | detection metrics only |
| `mavvid` | **yes** | no | no | detection metrics only |
| `antiuav` | no | yes | **no — single target** | MOTA/HOTA only; IDF1 and IDSW are vacuous |
| `mmuav` | no | yes | **yes** | everything, including IDF1 |

The subtle row is `antiuav`. It *has* track ids, so a naive `has_track_ids` check passes —
but every box is id 1, so every identity trivially matches every other. IDF1 would report a
perfect score for a tracker doing nothing. That is why `capabilities.py` derives an explicit
contract per source rather than trusting a single boolean, and why `track-eval` **warns** on
single-target ground truth instead of refusing it.

Only `mmuav` makes IDF1 meaningful, and it is Baidu-Pan-only with an interactive captcha —
not something to attempt live. The trackers themselves run fine on `dvb`; there is simply
nothing to score them against.

In [ ]:
!anti-uav track-eval --run artifacts/runs/yolo11n/dvb --dataset dvb

## 9. Inference and the rule layer

**This project's own trackers, not ultralytics'.** Version 8.4 ships no `track` task —
`persist` and `tracker` are not in the predict config, and `model.predict(..., persist=True)`
raises `SyntaxError`. Owning SORT / ByteTrack / BoT-SORT buys something more valuable: the
live path and the `track-eval` harness share **one** association implementation, so a number
from the harness describes what the pipeline actually does.

### Why inference is permissive and alerting is strict

Two different stages, two different jobs:

| | threshold | why |
| --- | --- | --- |
| detector | `conf_threshold: 0.25` | maximise **recall** — the tracker needs the low-confidence frames to carry motion evidence through a dip |
| rules | `confidence.initiate: 0.60` | maximise **precision** — buy it *after* tracking, using persistence, kinematics and cross-camera agreement |

Raise the detector threshold to 0.60 and the tracker starves. `predict` therefore prints the
per-detection table rather than a count, because 40 detections at conf 0.25 and 3 at 0.8 are
completely different situations.

### The five rule gates

`configs/rules/drone_rules.yaml`, in order: **confidence** (with hysteresis), **persistence**,
**kinematics**, **spatial**, **cross-camera**. Every threshold is tunable without a retrain.

Two of them return `skipped` rather than `passing` until the site is calibrated —
`spatial.horizon_y` and `ground_plane_z_m` are null, and they are per-camera properties.
That distinction is the point of the layer: **a missing input must never read as a satisfied
rule.** Only `FAIL` blocks an alert; an uncalibrated site runs three of five gates and says
plainly which two it skipped and what to configure.

Note `cross_camera.min_cameras: 2`. One view cannot distinguish a drone from a bird with
enough confidence to page anyone — so a track with no global identity is a **failure**, not a
skip. This is why cross-camera fusion is load-bearing for correctness rather than a nice
extra.

In [ ]:
# Grab a real frame from the interim index so the cell is self-contained. Swap --source for
# a video clip to show tracker behaviour over time - that is the more convincing demo, since
# a single frame cannot show a track being held through a dip or broken by an occlusion.
import glob

frames = sorted(
    glob.glob(f"{os.environ['ANTI_UAV_DATA_DIR']}/data/interim/dvb/*/frames/*/rgb/*.jpg")
)
print(f"{len(frames):,} interim frames available")

!anti-uav predict --run artifacts/runs/yolo11n/dvb --source {frames[len(frames)//2]} --tracker botsort --save /kaggle/tmp/preds

In [ ]:
!anti-uav rules show
!anti-uav rules validate

## 10. Keep the evidence

`artifacts/` lives under `/kaggle/tmp` and is **not** autosaved — the weights would vanish with
the session. Copy the record out to `/kaggle/working`, which is. That single step is the
difference between a demo and a result you can point at afterwards.

A run directory is the record: `best.pt`, `results.csv`, `args.yaml` and a metadata file.
**Copy it, do not regenerate it.** `data/processed/dvb/build_report.json` is worth keeping too
— it is what proves which sources were actually present, and `build` will happily skip a
missing one and still name the combo after it. A combo name is not evidence of dataset
membership.

In [ ]:
import shutil
from pathlib import Path

SRC = Path(os.environ["ANTI_UAV_DATA_DIR"])
DST = Path("/kaggle/working/artifacts")

for _rel in ("runs", "exports", "tracks"):
    _src = SRC / "artifacts" / _rel
    if _src.exists():
        shutil.copytree(_src, DST / _rel, dirs_exist_ok=True)
        print(f"saved artifacts/{_rel}")

_report = SRC / "data" / "processed" / "dvb" / "build_report.json"
if _report.is_file():
    (DST / "dvb_build_report.json").write_text(_report.read_text())
    print("saved dvb_build_report.json")

In [ ]:
!anti-uav list-runs

## What this notebook deliberately does not do

**`anti-uav serve`** — the operator console and JSON API. Kaggle tears the session down when
the last cell finishes and does not forward a listening port, so a long-lived server has
nowhere to live. Run it locally against the copied `artifacts/` — the UI reads runs off disk,
so the copied record is all it needs.

**TensorRT export** — the engine is normally built on the target device, not on a training
box, so `deploy render` *warns* rather than fails when the engine is missing and prints the
exact `trtexec` command. ONNX export works if you want it.

**The other 12 runs** — the matrix is 2 models × 7 combos; this notebook does 2 of the 14.
T4×2 proves the pipeline; it does not sweep it. On the 5090 the full matrix is hours; here it
is days.

**Tracking metrics** — see section 8. Only `mmuav` makes them meaningful, and it is
Baidu-Pan-only behind an interactive captcha.

---

## What two runs do and do not establish

Four public datasets, two architectures, one combo, one site model. These runs answer
**"do both families train on this data, which rejects birds better, and does the 42 M model buy
anything over the 2.6 M one on this dataset"**.

They do not answer "how well does this detect an unseen drone type at 40 m against a cluttered
sky". Four datasets is a design choice made under a time budget, not a claim of coverage.
**Treat the ranking as a ranking and the absolute numbers as specific to these four sources.**

The strongest claim here is a negative one, which is the useful kind: whether the detector
fires on birds. Everything finer needs the full matrix on real hardware — and the one
comparison that would actually justify RT-DETR's cost, `mmuav` at 12 px tiled, needs the
Baidu-Pan transfer.